# Data Governance — Lineage Tracking

Traces every data record (UUID) across all pipeline zones:

**Landing → Trusted → Exploitation → Milvus embeddings**

Shows zone presence, generated assets, transformation chain, and
pipeline completeness (6 stages total).

Prerequisites: MinIO running, at least one zone has processed data.

## Environment setup

In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /Users/arman/Desktop/2026-sprinrg/BDM/Cymatics/BDM-Cymatics


## MinIO connection — self-contained client setup

In [2]:
from minio import Minio

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "password")
MINIO_SECURE = os.environ.get("MINIO_SECURE", "false").lower() == "true"

LANDING_BUCKET = os.environ.get("LANDING_ZONE_BUCKET", "landing-zone")
TRUSTED_BUCKET = os.environ.get("TRUSTED_ZONE_BUCKET", "trusted-zone")
EXPLOITATION_BUCKET = os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone")

METADATA_KEY = "metadata/observations.csv"

def create_minio_client():
    return Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                 secret_key=MINIO_SECRET_KEY, secure=MINIO_SECURE)

minio_client = create_minio_client()
print(f"MinIO connected: {MINIO_ENDPOINT}")

MinIO connected: localhost:9000


## Step 1: Load metadata CSVs from all three MinIO zones

In [3]:
import io
import pandas as pd

def _load_csv_safe(bucket):
    try:
        resp = minio_client.get_object(bucket, METADATA_KEY)
        data = resp.read(); resp.close(); resp.release_conn()
        return pd.read_csv(io.BytesIO(data))
    except Exception:
        return pd.DataFrame()

landing_df = _load_csv_safe(LANDING_BUCKET)
trusted_df = _load_csv_safe(TRUSTED_BUCKET)
exploit_df = _load_csv_safe(EXPLOITATION_BUCKET)

print(f"  Landing:      {len(landing_df)} rows")
print(f"  Trusted:      {len(trusted_df)} rows")
print(f"  Exploitation: {len(exploit_df)} rows")

  Landing:      406 rows
  Trusted:      406 rows
  Exploitation: 406 rows


## Step 2: Index each zone's rows by UUID for fast lookup

In [4]:
def _index_by_uuid(df):
    idx = {}
    if not df.empty and "uuid" in df.columns:
        for _, row in df.iterrows():
            uid = str(row.get("uuid", "")).strip()
            if uid:
                idx[uid] = row.to_dict()
    return idx

landing_idx = _index_by_uuid(landing_df)
trusted_idx = _index_by_uuid(trusted_df)
exploit_idx = _index_by_uuid(exploit_df)

print(f"  Indexed: {len(landing_idx)} landing, {len(trusted_idx)} trusted, {len(exploit_idx)} exploitation")

  Indexed: 406 landing, 406 trusted, 406 exploitation


## Step 3: Check which UUIDs have embeddings in the 3 Milvus collections

In [5]:
milvus_uuids = {}
try:
    from pymilvus import MilvusClient
    MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")
    mc = MilvusClient(uri=MILVUS_URI)
    for coll in ["sound_audio_embeddings", "sound_text_embeddings", "sound_cymatics_embeddings"]:
        if not mc.has_collection(coll):
            milvus_uuids[coll] = set()
            continue
        stats = mc.get_collection_stats(coll)
        count = int(stats.get("row_count", 0))
        if count == 0:
            milvus_uuids[coll] = set()
            continue
        results = mc.query(collection_name=coll, filter="", output_fields=["uuid"], limit=count)
        milvus_uuids[coll] = {r["uuid"] for r in results}
    print(f"  Audio embeddings:    {len(milvus_uuids.get('sound_audio_embeddings', set()))} UUIDs")
    print(f"  Text embeddings:     {len(milvus_uuids.get('sound_text_embeddings', set()))} UUIDs")
    print(f"  Cymatics embeddings: {len(milvus_uuids.get('sound_cymatics_embeddings', set()))} UUIDs")
except Exception as e:
    print(f"  Milvus not reachable: {e}")

  Audio embeddings:    406 UUIDs
  Text embeddings:     406 UUIDs
  Cymatics embeddings: 406 UUIDs


## Step 4: Union all UUIDs and build lineage per record

In [6]:
audio_emb = milvus_uuids.get("sound_audio_embeddings", set())
text_emb = milvus_uuids.get("sound_text_embeddings", set())
cymatics_emb = milvus_uuids.get("sound_cymatics_embeddings", set())

all_uuids = sorted(
    set(landing_idx) | set(trusted_idx) | set(exploit_idx)
    | audio_emb | text_emb | cymatics_emb
)
print(f"Building lineage for {len(all_uuids)} unique records...")

lineage = []
for uid in all_uuids:
    landing = landing_idx.get(uid)
    trusted = trusted_idx.get(uid)
    exploit = exploit_idx.get(uid)

    source = str(landing.get("source", "")) or "—" if landing else "—"
    category = str(landing.get("category", "")) or "—" if landing else "—"
    if trusted and category == "—":
        category = str(trusted.get("category", "")) or "—"

    in_l = uid in landing_idx
    in_t = uid in trusted_idx
    in_e = uid in exploit_idx

    stages = []
    if in_l: stages.append("landing")
    if in_t: stages.append("trusted")
    if in_e: stages.append("exploitation")

    transformations = []
    if in_l: transformations.append(f"Ingested via {source} → landing-zone audio")
    if in_t:
        pv = str(trusted.get("processing_version", "")) if trusted else ""
        transformations.append(f"Spark QA → trusted-zone (image + video + peak, v{pv})")
    if in_e:
        fv = str(exploit.get("feature_version", "")) if exploit else ""
        transformations.append(f"Spark spectral + Python MFCCs → exploitation-zone (v{fv})")
    if uid in audio_emb: transformations.append("PANNs CNN14 → audio embedding (2048-dim)")
    if uid in text_emb: transformations.append("all-MiniLM-L6-v2 → text embedding (384-dim)")
    if uid in cymatics_emb: transformations.append("CLIP ViT-B/32 → cymatics embedding (512-dim)")

    completed = int(in_l) + int(in_t) + int(in_e) + int(uid in audio_emb) + int(uid in text_emb) + int(uid in cymatics_emb)
    completeness = completed / 6

    lineage.append({
        "uuid": uid, "source": source, "category": category,
        "in_landing": in_l, "in_trusted": in_t, "in_exploitation": in_e,
        "has_audio_embedding": uid in audio_emb,
        "has_text_embedding": uid in text_emb,
        "has_cymatics_embedding": uid in cymatics_emb,
        "stages_completed": stages, "transformations": transformations,
        "completeness": completeness,
    })

print(f"  Built {len(lineage)} lineage records.")

Building lineage for 406 unique records...
  Built 406 lineage records.


## Display lineage — summary + per-record transformation chain

In [7]:
full = sum(1 for r in lineage if r["completeness"] == 1.0)
partial = sum(1 for r in lineage if 0 < r["completeness"] < 1.0)
landing_only = sum(1 for r in lineage if r["in_landing"] and not r["in_trusted"] and not r["in_exploitation"])

print(f"\n{'=' * 62}")
print(f"  Data Lineage — {len(lineage)} records")
print(f"{'─' * 62}")
print(f"  Full pipeline (6/6):   {full}")
print(f"  Partial:               {partial}")
print(f"  Landing only:          {landing_only}")
print(f"{'─' * 62}")

# Show first 10 records.
for i, rec in enumerate(lineage[:10]):
    comp = rec["completeness"]
    stages = " → ".join(rec["stages_completed"]) or "—"
    bar_len = 20
    filled = int(comp * bar_len)
    bar = "█" * filled + "░" * (bar_len - filled)
    print(f"\n  {i+1}. {rec['uuid'][:24]}…")
    print(f"     Category:   {rec['category']}")
    print(f"     Source:     {rec['source']}")
    print(f"     Stages:     {stages}")
    print(f"     Progress:   [{bar}] {comp:.0%}")
    print(f"     Chain:")
    for t in rec["transformations"]:
        print(f"       → {t}")

if len(lineage) > 10:
    print(f"\n  ... and {len(lineage) - 10} more records.")
print(f"\n{'=' * 62}")


  Data Lineage — 406 records
──────────────────────────────────────────────────────────────
  Full pipeline (6/6):   406
  Partial:               0
  Landing only:          0
──────────────────────────────────────────────────────────────

  1. 00075442-8b5e-4bbd-ad3f-…
     Category:   church bells
     Source:     Freesound
     Stages:     landing → trusted → exploitation
     Progress:   [████████████████████] 100%
     Chain:
       → Ingested via Freesound → landing-zone audio
       → Spark QA → trusted-zone (image + video + peak, v2.0.0)
       → Spark spectral + Python MFCCs → exploitation-zone (v1.0.0)
       → PANNs CNN14 → audio embedding (2048-dim)
       → all-MiniLM-L6-v2 → text embedding (384-dim)
       → CLIP ViT-B/32 → cymatics embedding (512-dim)

  2. 00724c25-5227-4833-8b71-…
     Category:   wind
     Source:     Freesound
     Stages:     landing → trusted → exploitation
     Progress:   [████████████████████] 100%
     Chain:
       → Ingested via Freesound → l

## Pipeline completeness summary

In [8]:
total = len(lineage)
if total > 0:
    in_l = sum(1 for r in lineage if r["in_landing"])
    in_t = sum(1 for r in lineage if r["in_trusted"])
    in_e = sum(1 for r in lineage if r["in_exploitation"])
    has_ae = sum(1 for r in lineage if r["has_audio_embedding"])
    has_te = sum(1 for r in lineage if r["has_text_embedding"])
    has_ce = sum(1 for r in lineage if r["has_cymatics_embedding"])

    print(f"\n{'─' * 62}")
    print("  Per-stage presence:")
    for label, count in [
        ("Landing zone", in_l), ("Trusted zone", in_t), ("Exploitation zone", in_e),
        ("Audio embeddings", has_ae), ("Text embeddings", has_te), ("Cymatics embeddings", has_ce),
    ]:
        pct = count / total
        filled = int(pct * 30)
        bar = "█" * filled + "░" * (30 - filled)
        print(f"    {label:<22} [{bar}] {count}/{total}")
    print(f"{'─' * 62}")
else:
    print("No lineage records found.")


──────────────────────────────────────────────────────────────
  Per-stage presence:
    Landing zone           [██████████████████████████████] 406/406
    Trusted zone           [██████████████████████████████] 406/406
    Exploitation zone      [██████████████████████████████] 406/406
    Audio embeddings       [██████████████████████████████] 406/406
    Text embeddings        [██████████████████████████████] 406/406
    Cymatics embeddings    [██████████████████████████████] 406/406
──────────────────────────────────────────────────────────────


## Save lineage to MinIO

In [ ]:
import json

payload = json.dumps(lineage, indent=2, default=str).encode("utf-8")
minio_client.put_object(
    EXPLOITATION_BUCKET,
    "governance/lineage.json",
    io.BytesIO(payload),
    length=len(payload),
    content_type="application/json",
)
print(f"Lineage saved: {EXPLOITATION_BUCKET}/governance/lineage.json ({len(payload)/1024:.1f} KB)")